# Foo

Prepare dataset and logic for a character-based auto-regressive language model, based on a small sampling of Shakespearean text.

* Download `tinyshakespeare/input.txt`
* Create vocabular of unique chars in `input.txt`
* Define mapping from char to int (encode), and from int to char (decode)
* Encode the entire text data of `input.txt`
* Split dataset into training and validation datasets
* Define function for obtaining a batch data from either training or validation dataset

In [1]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()

# download the tiny shakespeare dataset
input_file_path = os.path.join(PROJECT_ROOT, 'input.txt')
if not os.path.exists(input_file_path):
    data_url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    with open(input_file_path, 'w') as f:
        f.write(requests.get(data_url).text)

In [2]:
with open(input_file_path, 'r') as f:
    text = f.read()
print(f"length of dataset in characters: {len(text)}")

length of dataset in characters: 1115394


In [3]:
# print first 1000 chars in the raw text
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [4]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("all the unique characters:", ''.join(chars))
print(f"vocab size: {vocab_size:,}")

all the unique characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab size: 65


In [5]:
# create mapping from chars to int
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [6]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"torch version is: {torch.__version__}")
print(f"GPU available? {torch.cuda.is_available()}, using {device}")

torch version is: 2.13.0+cu130
GPU available? True, using cuda


In [7]:
# encode the entire text dataset and store into torch.Tensor
data = torch.tensor(encode(text), dtype=torch.int)

print(f"{type(data)}")
print(data.shape, data.dtype)

# print the first 1000 int in the dataset
print(data[:1000])

<class 'torch.Tensor'>
torch.Size([1115394]) torch.int32
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1,

In [8]:
# split up the data into train and validation datasets
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

## `T`, the time dimension 

The time dimension spans the length of a single context.

In a single batch and across the dimension of time (positions `0` through `max_context_size-1`, we will train our character-based language model such that it will get used to seeing contexts as short as one character up to the maximum `max_context_size` characters.

Our character-based language model will look at all of the chars of a given context (looking at chars from preceding moments, as it were) to predict the next character in the sequence.

_In the video, Karpathy uses the name `block_size`, but since `Block` is an integral part of the Transformer architecture, we will replace `block_size` with `max_context_size` to avoid confusion._

In [9]:
# hyperparameters
batch_size = 4          # how many independent sequences to parallel-process?
#block_size = 8         # ... renaming, since "Block" is part of Tranformer architecture
max_context_size = 8    # what is the maximum context length for predictions?

In [10]:
train_data[:max_context_size]

tensor([18, 47, 56, 57, 58,  1, 15, 47], dtype=torch.int32)

In [11]:
x = train_data[:max_context_size]
y = train_data[1:max_context_size+1]
for t in range(max_context_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is: {context.tolist()}, the target is: {target}")

when input is: [18], the target is: 47
when input is: [18, 47], the target is: 56
when input is: [18, 47, 56], the target is: 57
when input is: [18, 47, 56, 57], the target is: 58
when input is: [18, 47, 56, 57, 58], the target is: 1
when input is: [18, 47, 56, 57, 58, 1], the target is: 15
when input is: [18, 47, 56, 57, 58, 1, 15], the target is: 47
when input is: [18, 47, 56, 57, 58, 1, 15, 47], the target is: 58


## `B`, the batch dimension

Processing one training example at a time is inefficient, while processing the entire dataset at once is usually impractical. A batch gives us a very useful middle ground.

In [12]:
def get_batch(split):
    # generate a small batch of data: inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - max_context_size, (batch_size,))
    x = torch.stack([data[i:i+max_context_size] for i in ix])
    y = torch.stack([data[i+1:i+max_context_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

In [13]:
torch.manual_seed(1337)

xb,yb = get_batch('train')

print('inputs:')
print(xb.shape)
print(xb)

print()

print('targets:')
print(yb.shape)
print(yb)

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]], device='cuda:0', dtype=torch.int32)

targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]], device='cuda:0', dtype=torch.int32)


In [14]:
for b in range(batch_size):                # batch dimension
    for t in range(max_context_size):      # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is: {context.tolist()}, the target is: {target}")
    print("---")

when input is: [24], the target is: 43
when input is: [24, 43], the target is: 58
when input is: [24, 43, 58], the target is: 5
when input is: [24, 43, 58, 5], the target is: 57
when input is: [24, 43, 58, 5, 57], the target is: 1
when input is: [24, 43, 58, 5, 57, 1], the target is: 46
when input is: [24, 43, 58, 5, 57, 1, 46], the target is: 43
when input is: [24, 43, 58, 5, 57, 1, 46, 43], the target is: 39
---
when input is: [44], the target is: 53
when input is: [44, 53], the target is: 56
when input is: [44, 53, 56], the target is: 1
when input is: [44, 53, 56, 1], the target is: 58
when input is: [44, 53, 56, 1, 58], the target is: 46
when input is: [44, 53, 56, 1, 58, 46], the target is: 39
when input is: [44, 53, 56, 1, 58, 46, 39], the target is: 58
when input is: [44, 53, 56, 1, 58, 46, 39, 58], the target is: 1
---
when input is: [52], the target is: 58
when input is: [52, 58], the target is: 1
when input is: [52, 58, 1], the target is: 58
when input is: [52, 58, 1, 58], th